# 10 — Retrieval & Ranking Evaluation

Evaluates the full multimodal retrieval and score-fusion pipeline using the real dataset.

## Evaluation methodology

We have **no human-labeled relevance judgments**. Instead we use **category-based pseudo-relevance**:

- A query product belongs to a category (e.g., `Footwear`).
- Any other product in the **same category** is considered **relevant**.
- A retrieved product is a **hit** if its category matches the query product's category.

This is a standard approach for evaluating retrieval systems without annotation. It measures **semantic consistency** — does the system surface products that are categorically similar to the query?

### Limitations (explicitly documented)
- Category-level relevance is coarse. A query for `"running shoes"` may retrieve `"formal shoes"` which is the same category but not truly relevant.
- The query product itself will be at rank 1 for image-only queries (similarity = 1.0) — this is expected and correct; we can exclude it or keep it depending on the metric interpretation. We **exclude** the query product from relevance for text/multimodal and **note** its presence for image queries.
- MRR measures where the first relevant product appears. With category-based relevance and a large number of positives per category, MRR is expected to be high.

## Metrics computed
| Metric | Description |
|---|---|
| **Precision@K** | Fraction of top-K results that are relevant |
| **Recall@K** | Fraction of all relevant products retrieved in top-K |
| **Hit Rate@K** | 1 if at least one relevant result in top-K, else 0 |
| **MRR** | Mean Reciprocal Rank — 1/rank of first relevant result |

## Query set
- One representative product per category is selected programmatically as a query.
- Multiple K values: 5, 10, 20.
- Three retrieval modes: text-only, image-only, text+image.
- Score fusion evaluated at multiple weight configurations.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import faiss
import torch
from pathlib import Path
from PIL import Image
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer
from collections import defaultdict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device : {DEVICE}")
print(f"faiss  : {faiss.__version__}")

device : cpu
faiss  : 1.15.0


## 2. Paths and Sanity Checks

In [2]:
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED    = PROJECT_ROOT / "data" / "processed"
FAISS_DIR    = PROCESSED / "faiss"

PRODUCTS_CSV      = PROCESSED / "products_ml_ready.csv"
TEXT_FAISS_PATH   = FAISS_DIR  / "text_index.faiss"
IMAGE_FAISS_PATH  = FAISS_DIR  / "image_index.faiss"
TEXT_MAPPING_CSV  = FAISS_DIR  / "text_index_mapping.csv"
IMAGE_MAPPING_CSV = FAISS_DIR  / "image_index_mapping.csv"

for p in [PRODUCTS_CSV, TEXT_FAISS_PATH, IMAGE_FAISS_PATH,
          TEXT_MAPPING_CSV, IMAGE_MAPPING_CSV]:
    assert p.exists(), f"Missing: {p}"
    print(f"  OK  {p.relative_to(PROJECT_ROOT)}")

  OK  data\processed\products_ml_ready.csv
  OK  data\processed\faiss\text_index.faiss
  OK  data\processed\faiss\image_index.faiss
  OK  data\processed\faiss\text_index_mapping.csv
  OK  data\processed\faiss\image_index_mapping.csv


## 3. Load Data

In [3]:
text_index  = faiss.read_index(str(TEXT_FAISS_PATH))
image_index = faiss.read_index(str(IMAGE_FAISS_PATH))

text_mapping_df  = pd.read_csv(TEXT_MAPPING_CSV)
image_mapping_df = pd.read_csv(IMAGE_MAPPING_CSV)

text_faiss_to_pid  = dict(zip(text_mapping_df["faiss_index"],  text_mapping_df["pid"]))
image_faiss_to_pid = dict(zip(image_mapping_df["faiss_index"], image_mapping_df["pid"]))

products_df     = pd.read_csv(PRODUCTS_CSV)
products_by_pid = products_df.set_index("pid")

N_PRODUCTS = len(products_df)
FAISS_DIM  = text_index.d

assert text_index.d  == 512
assert image_index.d == 512
assert text_index.ntotal  == N_PRODUCTS
assert image_index.ntotal == N_PRODUCTS

# Category → set of PIDs (for relevance judgment)
category_to_pids = products_df.groupby("main_category")["pid"].apply(set).to_dict()

print(f"Products       : {N_PRODUCTS}")
print(f"FAISS dim      : {FAISS_DIM}")
print(f"Categories     : {len(category_to_pids)}")
for cat, pids in sorted(category_to_pids.items(), key=lambda x: -len(x[1])):
    print(f"  {cat:35s}: {len(pids):4d} products")

Products       : 4681
FAISS dim      : 512
Categories     : 13
  Clothing                           :  814 products
  Jewellery                          :  700 products
  Footwear                           :  500 products
  Mobiles & Accessories              :  450 products
  Automotive                         :  400 products
  Home Decor & Festive Needs         :  350 products
  Beauty and Personal Care           :  300 products
  Home Furnishing                    :  300 products
  Computers                          :  250 products
  Kitchen & Dining                   :  250 products
  Tools & Hardware                   :  150 products
  Baby Care                          :  139 products
  Watches                            :   78 products


## 4. Load CLIP Model

In [4]:
MODEL_NAME = "openai/clip-vit-base-patch32"
print(f"Loading {MODEL_NAME} ...")
clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model.eval()
print(f"Model ready on : {DEVICE}")

Loading openai/clip-vit-base-patch32 ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model ready on : cpu


## 5. Retrieval + Ranking Layer (reused from Notebooks 08 & 09)

In [5]:
def _l2_normalize(vec):
    norm = np.linalg.norm(vec, axis=1, keepdims=True)
    return vec / np.clip(norm, 1e-10, None)

def encode_text(query):
    if not query or not query.strip():
        raise ValueError("Empty text query.")
    toks = clip_tokenizer([query.strip()], return_tensors="pt",
                           padding=True, truncation=True, max_length=77)
    toks = {k: v.to(DEVICE) for k, v in toks.items()}
    with torch.no_grad():
        out = clip_model.text_model(**toks)
        emb = clip_model.text_projection(out.pooler_output)
    return _l2_normalize(emb.cpu().float().numpy())

def encode_image(image_path):
    path = Path(image_path)
    if not path.is_absolute():
        path = (NOTEBOOK_DIR / path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")
    img = Image.open(path).convert("RGB")
    inputs = clip_processor(images=[img], return_tensors="pt")
    pv = inputs["pixel_values"].to(DEVICE)
    with torch.no_grad():
        vis = clip_model.vision_model(pixel_values=pv)
        emb = clip_model.visual_projection(vis.pooler_output)
    return _l2_normalize(emb.cpu().float().numpy())

def _faiss_search(index, faiss_to_pid, query_vec, top_k):
    scores, indices = index.search(query_vec.astype(np.float32), top_k)
    results = []
    for fidx, score in zip(indices[0], scores[0]):
        if fidx == -1: continue
        pid = faiss_to_pid.get(int(fidx))
        if pid is None or pid not in products_by_pid.index: continue
        results.append({"pid": pid, "score": float(score)})
    return results

def _normalize_col(series):
    """Min-max normalize a Series; NaN → 0.0."""
    vals = series.dropna()
    if vals.empty:
        return series.fillna(0.0)
    vmin, vmax = vals.min(), vals.max()
    spread = vmax - vmin
    if spread < 1e-10:
        return series.apply(lambda x: 1.0 if pd.notna(x) else 0.0)
    return series.apply(lambda x: float((x - vmin) / spread) if pd.notna(x) else 0.0)

def retrieve_and_rank(text_query=None, image_path=None,
                      retrieval_k=50, top_k=20,
                      text_weight=0.5, image_weight=0.5):
    """
    Full pipeline → returns list of PIDs in ranked order (top_k items).
    """
    has_text  = text_query  is not None and str(text_query).strip()  != ""
    has_image = image_path  is not None and str(image_path).strip()  != ""
    if not has_text and not has_image:
        raise ValueError("Need at least one of text_query or image_path.")

    t_rows, i_rows = [], []
    if has_text:
        t_rows = _faiss_search(text_index,  text_faiss_to_pid,  encode_text(text_query), retrieval_k)
    if has_image:
        i_rows = _faiss_search(image_index, image_faiss_to_pid, encode_image(image_path), retrieval_k)

    t_df = pd.DataFrame(t_rows).rename(columns={"score": "text_score"})  if t_rows else pd.DataFrame(columns=["pid","text_score"])
    i_df = pd.DataFrame(i_rows).rename(columns={"score": "image_score"}) if i_rows else pd.DataFrame(columns=["pid","image_score"])

    if not t_df.empty and not i_df.empty:
        merged = t_df.merge(i_df, on="pid", how="outer")
    elif not t_df.empty:
        merged = t_df.copy(); merged["image_score"] = np.nan
    else:
        merged = i_df.copy(); merged["text_score"] = np.nan

    merged["norm_text"]  = _normalize_col(merged["text_score"])
    merged["norm_image"] = _normalize_col(merged["image_score"])
    merged["final"]      = text_weight * merged["norm_text"] + image_weight * merged["norm_image"]

    ranked = merged.sort_values("final", ascending=False).head(top_k)
    return ranked["pid"].tolist()

print("Retrieval + ranking layer loaded.")

Retrieval + ranking layer loaded.


## 6. Build Evaluation Query Set

### Query design

We create **N queries per category** by sampling real products from the dataset. Each query product provides:
- A **text query** derived from its `combined_text_clean` field (first 60 chars — short, realistic).
- An **image query** using its `image_path`.

Relevant products = all products in the **same category** (excluding the query product itself).

We use **3 products per category** (sampled with a fixed seed for reproducibility), giving ~39 evaluation queries.

In [6]:
QUERIES_PER_CATEGORY = 3
RANDOM_SEED          = 42

rng = np.random.default_rng(RANDOM_SEED)

query_set = []  # list of dicts

for category, pids in sorted(category_to_pids.items()):
    pid_list = sorted(pids)  # deterministic order before sampling
    n_sample = min(QUERIES_PER_CATEGORY, len(pid_list))
    chosen   = rng.choice(pid_list, size=n_sample, replace=False)

    for pid in chosen:
        row = products_by_pid.loc[pid]
        # Short text query from the cleaned combined text
        text_q = " ".join(str(row["combined_text_clean"]).split()[:12])
        query_set.append({
            "pid"           : pid,
            "category"      : category,
            "text_query"    : text_q,
            "image_path"    : row["image_path"],
            "n_relevant"    : len(pids) - 1,  # exclude query product itself
        })

query_df = pd.DataFrame(query_set)
print(f"Total queries      : {len(query_df)}")
print(f"Queries per cat    : {QUERIES_PER_CATEGORY}")
print(f"Categories covered : {query_df['category'].nunique()}")
print()
print(query_df[["category","pid","n_relevant","text_query"]].to_string(index=False))

Total queries      : 39
Queries per cat    : 3
Categories covered : 13

                  category              pid  n_relevant                                                                                     text_query
                Automotive CRTECN2RQJGXB54U         399                                product: allure auto cm 1458 car mat honda new city | category:
                Automotive BLKEBEUNFFPSFKHE         399 product: eshopitude disk break lock zargv1097-hondainterceptor u lock | category: automotive |
                Automotive SUDEGGATV7QB9GV9         399                       product: adroitz exclusive barbie doll sunshade / curtain (set of 2) for
                 Baby Care BTWEDRTHRY83HXYY         138                       product: neyth cotton set of towels | category: baby care | subcategory:
                 Baby Care STIE7VAYP6GYJRPQ         138          product: elite collection medium acrylic sticker | category: baby care | subcategory:
                 Baby 

## 7. Metric Functions

All metrics are computed per-query then averaged (macro average across queries).

In [7]:
def precision_at_k(retrieved_pids, relevant_pids, k):
    """Fraction of top-K retrieved that are relevant."""
    top_k = retrieved_pids[:k]
    hits  = sum(1 for p in top_k if p in relevant_pids)
    return hits / k

def recall_at_k(retrieved_pids, relevant_pids, k):
    """Fraction of all relevant products that appear in top-K."""
    if not relevant_pids:
        return 0.0
    top_k = retrieved_pids[:k]
    hits  = sum(1 for p in top_k if p in relevant_pids)
    return hits / len(relevant_pids)

def hit_rate_at_k(retrieved_pids, relevant_pids, k):
    """1 if at least one relevant product is in top-K, else 0."""
    return float(any(p in relevant_pids for p in retrieved_pids[:k]))

def reciprocal_rank(retrieved_pids, relevant_pids):
    """1/rank of the first relevant product (0 if none found)."""
    for rank, pid in enumerate(retrieved_pids, start=1):
        if pid in relevant_pids:
            return 1.0 / rank
    return 0.0

def compute_metrics(retrieved_pids, relevant_pids, k_values):
    """Compute all metrics for a single query at multiple K values."""
    result = {}
    for k in k_values:
        result[f"P@{k}"]  = precision_at_k(retrieved_pids, relevant_pids, k)
        result[f"R@{k}"]  = recall_at_k(retrieved_pids, relevant_pids, k)
        result[f"HR@{k}"] = hit_rate_at_k(retrieved_pids, relevant_pids, k)
    result["RR"] = reciprocal_rank(retrieved_pids, relevant_pids)
    return result

K_VALUES = [5, 10, 20]
print(f"Metrics: Precision@K, Recall@K, Hit Rate@K for K in {K_VALUES}, plus MRR")

Metrics: Precision@K, Recall@K, Hit Rate@K for K in [5, 10, 20], plus MRR


## 8. Evaluation Runner

Runs all queries for a given retrieval mode and collects per-query metrics.

In [8]:
def run_evaluation(query_df, mode, retrieval_k=50, top_k=20,
                   text_weight=0.5, image_weight=0.5):
    """
    Run full evaluation over query_df for a given mode.

    mode: 'text', 'image', or 'multimodal'

    Returns
    -------
    pd.DataFrame — one row per query with all metrics.
    """
    records = []

    for _, q in query_df.iterrows():
        query_pid  = q["pid"]
        category   = q["category"]
        relevant   = category_to_pids[category] - {query_pid}  # exclude self

        try:
            if mode == "text":
                ranked = retrieve_and_rank(
                    text_query   = q["text_query"],
                    retrieval_k  = retrieval_k,
                    top_k        = top_k,
                    text_weight  = 1.0,
                    image_weight = 0.0,
                )
            elif mode == "image":
                ranked = retrieve_and_rank(
                    image_path   = q["image_path"],
                    retrieval_k  = retrieval_k,
                    top_k        = top_k,
                    text_weight  = 0.0,
                    image_weight = 1.0,
                )
                # For image-only, rank 1 is always self (score=1.0); exclude
                ranked = [p for p in ranked if p != query_pid]
            elif mode == "multimodal":
                ranked = retrieve_and_rank(
                    text_query   = q["text_query"],
                    image_path   = q["image_path"],
                    retrieval_k  = retrieval_k,
                    top_k        = top_k,
                    text_weight  = text_weight,
                    image_weight = image_weight,
                )
                ranked = [p for p in ranked if p != query_pid]
            else:
                raise ValueError(f"Unknown mode: {mode}")

        except Exception as e:
            print(f"  WARN: query {query_pid} ({mode}) failed: {e}")
            continue

        metrics = compute_metrics(ranked, relevant, K_VALUES)
        records.append({
            "pid"       : query_pid,
            "category"  : category,
            "n_relevant": len(relevant),
            "n_retrieved": len(ranked),
            **metrics,
        })

    return pd.DataFrame(records)


print("run_evaluation defined.")

run_evaluation defined.


## 9. Run Evaluation — Text-Only Retrieval

In [9]:
RETRIEVAL_K = 50
TOP_K       = 20

print("Running text-only evaluation ...")
eval_text = run_evaluation(query_df, mode="text",
                           retrieval_k=RETRIEVAL_K, top_k=TOP_K)

print(f"Queries evaluated: {len(eval_text)}")
pd.set_option("display.float_format", "{:.4f}".format)
print("\nPer-category mean metrics (text-only):")
cat_metrics_text = eval_text.groupby("category")[
    [f"P@{k}" for k in K_VALUES] +
    [f"R@{k}" for k in K_VALUES] +
    [f"HR@{k}" for k in K_VALUES] + ["RR"]
].mean().round(4)
print(cat_metrics_text.to_string())

Running text-only evaluation ...


Queries evaluated: 39

Per-category mean metrics (text-only):
                              P@5   P@10   P@20    R@5   R@10   R@20   HR@5  HR@10  HR@20     RR
category                                                                                        
Automotive                 0.8667 0.9333 0.9500 0.0109 0.0234 0.0476 1.0000 1.0000 1.0000 0.8333
Baby Care                  0.8000 0.7333 0.6833 0.0290 0.0531 0.0990 1.0000 1.0000 1.0000 1.0000
Beauty and Personal Care   0.4667 0.5000 0.4000 0.0078 0.0167 0.0268 0.6667 0.6667 0.6667 0.3333
Clothing                   0.8000 0.9000 0.9167 0.0049 0.0111 0.0226 1.0000 1.0000 1.0000 0.8333
Computers                  0.5333 0.5667 0.5333 0.0107 0.0228 0.0428 0.6667 0.6667 1.0000 0.3500
Footwear                   1.0000 0.9667 0.9500 0.0100 0.0194 0.0381 1.0000 1.0000 1.0000 1.0000
Home Decor & Festive Needs 0.5333 0.6000 0.5833 0.0076 0.0172 0.0334 0.6667 0.6667 0.6667 0.3333
Home Furnishing            0.8667 0.9000 0.9500 0.0145 0.0301 0.0

## 10. Run Evaluation — Image-Only Retrieval

In [10]:
print("Running image-only evaluation ...")
eval_image = run_evaluation(query_df, mode="image",
                            retrieval_k=RETRIEVAL_K, top_k=TOP_K)

print(f"Queries evaluated: {len(eval_image)}")
print("\nPer-category mean metrics (image-only):")
cat_metrics_image = eval_image.groupby("category")[
    [f"P@{k}" for k in K_VALUES] +
    [f"R@{k}" for k in K_VALUES] +
    [f"HR@{k}" for k in K_VALUES] + ["RR"]
].mean().round(4)
print(cat_metrics_image.to_string())

Running image-only evaluation ...


Queries evaluated: 39

Per-category mean metrics (image-only):
                              P@5   P@10   P@20    R@5   R@10   R@20   HR@5  HR@10  HR@20     RR
category                                                                                        
Automotive                 0.8667 0.8333 0.6167 0.0109 0.0209 0.0309 1.0000 1.0000 1.0000 1.0000
Baby Care                  0.6000 0.4000 0.2500 0.0217 0.0290 0.0362 0.6667 0.6667 0.6667 0.6667
Beauty and Personal Care   0.4000 0.4000 0.3667 0.0067 0.0134 0.0245 0.6667 0.6667 0.6667 0.4167
Clothing                   1.0000 1.0000 0.9333 0.0062 0.0123 0.0230 1.0000 1.0000 1.0000 1.0000
Computers                  0.4667 0.4333 0.3667 0.0094 0.0174 0.0295 0.6667 0.6667 0.6667 0.5000
Footwear                   1.0000 1.0000 0.9500 0.0100 0.0200 0.0381 1.0000 1.0000 1.0000 1.0000
Home Decor & Festive Needs 0.6667 0.6333 0.5167 0.0096 0.0181 0.0296 0.6667 0.6667 0.6667 0.6667
Home Furnishing            1.0000 0.9667 0.9167 0.0167 0.0323 0.

## 11. Run Evaluation — Multimodal Retrieval (Balanced 0.5 / 0.5)

In [11]:
print("Running multimodal evaluation (text=0.5, image=0.5) ...")
eval_mm = run_evaluation(query_df, mode="multimodal",
                         retrieval_k=RETRIEVAL_K, top_k=TOP_K,
                         text_weight=0.5, image_weight=0.5)

print(f"Queries evaluated: {len(eval_mm)}")
print("\nPer-category mean metrics (multimodal 0.5/0.5):")
cat_metrics_mm = eval_mm.groupby("category")[
    [f"P@{k}" for k in K_VALUES] +
    [f"R@{k}" for k in K_VALUES] +
    [f"HR@{k}" for k in K_VALUES] + ["RR"]
].mean().round(4)
print(cat_metrics_mm.to_string())

Running multimodal evaluation (text=0.5, image=0.5) ...


Queries evaluated: 39

Per-category mean metrics (multimodal 0.5/0.5):
                              P@5   P@10   P@20    R@5   R@10   R@20   HR@5  HR@10  HR@20     RR
category                                                                                        
Automotive                 1.0000 0.9667 0.9167 0.0125 0.0242 0.0459 1.0000 1.0000 1.0000 1.0000
Baby Care                  0.8000 0.7667 0.6667 0.0290 0.0556 0.0966 1.0000 1.0000 1.0000 1.0000
Beauty and Personal Care   0.6000 0.6000 0.4667 0.0100 0.0201 0.0312 1.0000 1.0000 1.0000 0.7500
Clothing                   1.0000 1.0000 0.9333 0.0062 0.0123 0.0230 1.0000 1.0000 1.0000 1.0000
Computers                  0.6000 0.6333 0.5167 0.0120 0.0254 0.0415 0.6667 0.6667 0.6667 0.6667
Footwear                   1.0000 1.0000 0.9500 0.0100 0.0200 0.0381 1.0000 1.0000 1.0000 1.0000
Home Decor & Festive Needs 0.6667 0.6667 0.5833 0.0096 0.0191 0.0334 0.6667 0.6667 0.6667 0.6667
Home Furnishing            1.0000 1.0000 0.9500 0.0167 0

## 12. Compare Retrieval Modes — Overall Macro Averages

In [12]:
metric_cols = ([f"P@{k}" for k in K_VALUES] +
               [f"R@{k}" for k in K_VALUES] +
               [f"HR@{k}" for k in K_VALUES] + ["RR"])

summary = pd.DataFrame({
    "Text-only"         : eval_text[metric_cols].mean(),
    "Image-only"        : eval_image[metric_cols].mean(),
    "Multimodal (0.5/0.5)": eval_mm[metric_cols].mean(),
}).T.round(4)

print("=" * 70)
print("OVERALL MACRO-AVERAGE METRICS — MODE COMPARISON")
print("=" * 70)
print(summary.to_string())

OVERALL MACRO-AVERAGE METRICS — MODE COMPARISON
                        P@5   P@10   P@20    R@5   R@10   R@20   HR@5  HR@10  HR@20     RR
Text-only            0.7590 0.8000 0.8000 0.0155 0.0326 0.0656 0.9231 0.9231 0.9487 0.7064
Image-only           0.7897 0.7513 0.6590 0.0161 0.0303 0.0532 0.8974 0.8974 0.8974 0.8654
Multimodal (0.5/0.5) 0.8718 0.8667 0.7987 0.0179 0.0356 0.0653 0.9487 0.9487 0.9487 0.9295


## 13. Weight Sensitivity Analysis

Evaluate multimodal fusion at several text/image weight configurations to find the most effective balance.

In [13]:
weight_configs = [
    ("Text only (1.0/0.0)",   1.0, 0.0),
    ("Text-heavy (0.7/0.3)",  0.7, 0.3),
    ("Balanced (0.5/0.5)",    0.5, 0.5),
    ("Image-heavy (0.3/0.7)", 0.3, 0.7),
    ("Image only (0.0/1.0)",  0.0, 1.0),
]

weight_results = {}

for label, tw, iw in weight_configs:
    print(f"  Evaluating {label} ...")
    if tw == 1.0 and iw == 0.0:
        df_w = eval_text   # already computed
    elif tw == 0.0 and iw == 1.0:
        df_w = eval_image  # already computed
    elif tw == 0.5 and iw == 0.5:
        df_w = eval_mm     # already computed
    else:
        df_w = run_evaluation(query_df, mode="multimodal",
                              retrieval_k=RETRIEVAL_K, top_k=TOP_K,
                              text_weight=tw, image_weight=iw)
    weight_results[label] = df_w[metric_cols].mean()

weight_summary = pd.DataFrame(weight_results).T.round(4)

print()
print("=" * 70)
print("WEIGHT SENSITIVITY — MACRO-AVERAGE METRICS")
print("=" * 70)
print(weight_summary.to_string())

  Evaluating Text only (1.0/0.0) ...
  Evaluating Text-heavy (0.7/0.3) ...


  Evaluating Balanced (0.5/0.5) ...
  Evaluating Image-heavy (0.3/0.7) ...


  Evaluating Image only (0.0/1.0) ...

WEIGHT SENSITIVITY — MACRO-AVERAGE METRICS
                         P@5   P@10   P@20    R@5   R@10   R@20   HR@5  HR@10  HR@20     RR
Text only (1.0/0.0)   0.7590 0.8000 0.8000 0.0155 0.0326 0.0656 0.9231 0.9231 0.9487 0.7064
Text-heavy (0.7/0.3)  0.8769 0.8667 0.8013 0.0180 0.0355 0.0657 0.9231 0.9231 0.9487 0.9247
Balanced (0.5/0.5)    0.8718 0.8667 0.7987 0.0179 0.0356 0.0653 0.9487 0.9487 0.9487 0.9295
Image-heavy (0.3/0.7) 0.8462 0.8385 0.7679 0.0174 0.0343 0.0631 0.9487 0.9487 0.9487 0.9124
Image only (0.0/1.0)  0.7897 0.7513 0.6590 0.0161 0.0303 0.0532 0.8974 0.8974 0.8974 0.8654


## 14. Per-Category Breakdown — Best Mode

Which categories benefit most from multimodal fusion vs. text-only or image-only?

In [14]:
# Compare P@10 and HR@10 per category across modes
compare_cols = ["P@10", "HR@10", "RR"]

cat_text  = eval_text.groupby("category")[compare_cols].mean().add_prefix("text_")
cat_image = eval_image.groupby("category")[compare_cols].mean().add_prefix("image_")
cat_mm    = eval_mm.groupby("category")[compare_cols].mean().add_prefix("mm_")

cat_compare = cat_text.join(cat_image).join(cat_mm).round(4)

# Determine best mode per category on P@10
cat_compare["best_P@10"] = cat_compare[["text_P@10","image_P@10","mm_P@10"]].idxmax(axis=1).str.replace("_P@10","")

print("Per-category comparison (P@10, HR@10, MRR):")
print(cat_compare.to_string())

Per-category comparison (P@10, HR@10, MRR):
                            text_P@10  text_HR@10  text_RR  image_P@10  image_HR@10  image_RR  mm_P@10  mm_HR@10  mm_RR best_P@10
category                                                                                                                         
Automotive                     0.9333      1.0000   0.8333      0.8333       1.0000    1.0000   0.9667    1.0000 1.0000        mm
Baby Care                      0.7333      1.0000   1.0000      0.4000       0.6667    0.6667   0.7667    1.0000 1.0000        mm
Beauty and Personal Care       0.5000      0.6667   0.3333      0.4000       0.6667    0.4167   0.6000    1.0000 0.7500        mm
Clothing                       0.9000      1.0000   0.8333      1.0000       1.0000    1.0000   1.0000    1.0000 1.0000     image
Computers                      0.5667      0.6667   0.3500      0.4333       0.6667    0.5000   0.6333    0.6667 0.6667        mm
Footwear                       0.9667      1.0

## 15. MRR Distribution

In [15]:
mrr_compare = pd.DataFrame({
    "Text-only"    : eval_text["RR"],
    "Image-only"   : eval_image["RR"],
    "Multimodal"   : eval_mm["RR"],
})

print("MRR descriptive statistics:")
print(mrr_compare.describe().round(4).to_string())
print()
print(f"Queries with RR=0 (no relevant in top-{TOP_K}):")
for col in mrr_compare.columns:
    n_zero = (mrr_compare[col] == 0).sum()
    print(f"  {col:25s}: {n_zero} / {len(mrr_compare)}")

MRR descriptive statistics:
       Text-only  Image-only  Multimodal
count    39.0000     39.0000     39.0000
mean      0.7064      0.8654      0.9295
std       0.3158      0.3287      0.2497
min       0.0000      0.0000      0.0000
25%       0.5000      1.0000      1.0000
50%       0.5000      1.0000      1.0000
75%       1.0000      1.0000      1.0000
max       1.0000      1.0000      1.0000

Queries with RR=0 (no relevant in top-20):
  Text-only                : 2 / 39
  Image-only               : 4 / 39
  Multimodal               : 2 / 39


## 16. Result Verification

In [16]:
def verify_eval(df, label):
    assert len(df) > 0,                         f"{label}: empty results"
    for col in metric_cols:
        assert df[col].between(0, 1).all(),     f"{label}: {col} out of [0,1]"
        assert df[col].notna().all(),            f"{label}: {col} has NaN"
    print(f"  {label}: {len(df)} queries, all metrics in [0,1], no NaN  ✓")

print("=== Verification ===")
verify_eval(eval_text,  "Text-only")
verify_eval(eval_image, "Image-only")
verify_eval(eval_mm,    "Multimodal")
print("All verifications passed.")

=== Verification ===
  Text-only: 39 queries, all metrics in [0,1], no NaN  ✓
  Image-only: 39 queries, all metrics in [0,1], no NaN  ✓
  Multimodal: 39 queries, all metrics in [0,1], no NaN  ✓
All verifications passed.


## 17. Final Report

In [17]:
best_mode_p10 = summary["P@10"].idxmax()
best_mode_mrr = summary["RR"].idxmax()
best_weight   = weight_summary["P@10"].idxmax()

print("=" * 65)
print("RETRIEVAL & RANKING EVALUATION — FINAL REPORT")
print("=" * 65)
print(f"Products in dataset       : {N_PRODUCTS}")
print(f"Categories                : {query_df['category'].nunique()}")
print(f"Total queries evaluated   : {len(query_df)}")
print(f"Queries per category      : {QUERIES_PER_CATEGORY}")
print(f"Relevance definition      : same main_category (pseudo-relevance)")
print(f"Retrieval pool size (K)   : {RETRIEVAL_K}")
print(f"Ranking cutoff (top_k)    : {TOP_K}")
print()
print("--- Overall Macro-Average Results ---")
print(summary[["P@5","P@10","R@10","HR@10","RR"]].to_string())
print()
print(f"Best mode by P@10        : {best_mode_p10}")
print(f"Best mode by MRR         : {best_mode_mrr}")
print()
print("--- Weight Sensitivity ---")
print(weight_summary[["P@10","HR@10","RR"]].to_string())
print(f"\nBest weight config by P@10: {best_weight}")
print()
print("--- Limitations ---")
print("1. Relevance = same category (coarse; no human labels available)")
print("2. Text queries derived from product's own text (favorable for text search)")
print("3. Image queries exclude self-retrieval (rank 1 always matches self)")
print("4. Recall@K is bounded by min(K, n_relevant)/n_relevant — large categories")
print("   make Recall@K artificially low")
print()
print("Status: COMPLETE")
print("Next stage: Backend API / Frontend")
print("=" * 65)

RETRIEVAL & RANKING EVALUATION — FINAL REPORT
Products in dataset       : 4681
Categories                : 13
Total queries evaluated   : 39
Queries per category      : 3
Relevance definition      : same main_category (pseudo-relevance)
Retrieval pool size (K)   : 50
Ranking cutoff (top_k)    : 20

--- Overall Macro-Average Results ---
                        P@5   P@10   R@10  HR@10     RR
Text-only            0.7590 0.8000 0.0326 0.9231 0.7064
Image-only           0.7897 0.7513 0.0303 0.8974 0.8654
Multimodal (0.5/0.5) 0.8718 0.8667 0.0356 0.9487 0.9295

Best mode by P@10        : Multimodal (0.5/0.5)
Best mode by MRR         : Multimodal (0.5/0.5)

--- Weight Sensitivity ---
                        P@10  HR@10     RR
Text only (1.0/0.0)   0.8000 0.9231 0.7064
Text-heavy (0.7/0.3)  0.8667 0.9231 0.9247
Balanced (0.5/0.5)    0.8667 0.9487 0.9295
Image-heavy (0.3/0.7) 0.8385 0.9487 0.9124
Image only (0.0/1.0)  0.7513 0.8974 0.8654

Best weight config by P@10: Text-heavy (0.7/0.3)

--- 